# BAT 157-month top-50 Seyfert-proxy AGN cutoff search

Test the **50 largest 14–195 keV `FLUX` values after selecting BAT classes 4 and 5**
(Seyfert I and Seyfert II) independently over **0.1–10 MeV (100–10,000 keV)**.
This is a catalog-only radio-quiet proxy: BAT does not identify radio-loud
Seyferts, so the selection can retain them. The YAML reference model uses the
catalog BAT-only photon index and an imposed 300 keV exponential cutoff.

For each source, the notebook tests the cutoff grid in ascending order,
starting at 100 keV. It runs all three-month Poisson realizations at a cutoff,
calculates their median uncut TS, applies the source/cutoff-specific 60° FOV
significance gain, and compares the result at the same three-month exposure. It stops testing that
source as soon as

$$TS_{\rm 3mo,FOV\ est}\approx
G^2\,\mathrm{median}(TS_{\rm 3mo,uncut}) \ge 25.$$

Here, $G\approx f_S/\sqrt{f_B}$ is the count-based FOV significance gain. If
the target is not reached, the source continues through 1000 keV. A cutoff
with any failed trial or an unusable FOV estimate is inconclusive and cannot
trigger early stopping. The next cutoff is still tested. Untested higher
cutoffs are omitted from result tables and shown as missing values in plots.

Sources are processed in decreasing BAT flux. If a source conclusively fails to
reach TS 25 over the full cutoff grid even after the 24-month extrapolation
$TS_{24\,\mathrm{mo}}\approx8TS_{3\,\mathrm{mo}}$, a later fainter
source is skipped when its astromodels index is equal or more negative. A
source that is undetected at three months but detectable at 24 months does
not establish a pruning boundary. A later source is still tested when its
index is less negative (numerically larger), since its harder spectrum may
compensate for its lower BAT flux. A skipped source does not trigger further
pruning. This is a requested heuristic:
sky-position-dependent response and FOV gain can differ between sources,
so flux and index alone do not mathematically guarantee non-detection.

At every cutoff, the normalization is recalculated to preserve the source's
catalog 14–195 keV energy flux. Thus changes in MeV flux and significance come
from the assumed cutoff, rather than from changing the BAT measurement.

The maximum run is 50 sources, 10 cutoffs, and 20 Poisson realizations per
source/cutoff pair: **10,000 likelihood trials**. Early stopping can reduce this
substantially. Use `SOURCE_NAMES`, `CUTOFF_VALUES_KEV`, and `N_REALIZATIONS`
below for a smoke test before a full run. Each deterministic source/cutoff
folded expectation is calculated once and reused for its realizations.

Each trial fits only source normalization and background rate. Other catalog
sources are absent from that trial. Detailed results remain in memory and are displayed. The notebook saves only a
compact evaluated-scenario checkpoint for recreating the final detection tables
and plots after a kernel restart. It does not save trial-level results, generated
source files, plots, response projections, or FOV energy-count tables.

## Generate the input model

From the sibling AGN project in the Python environment containing astromodels:

```bash
cd /path/to/AGN-HE-detectability
python generate_bat157_top50_model.py --overwrite
```

The generator reads `BAT_157m_catalog_20250525.csv`, handles its padded headers
and units row, keeps only `CL2 = 4` or `CL2 = 5`, ranks the retained sources only
by `FLUX`, and writes `cosi_bat_157m_top50_models.yaml`.


In [1]:
import os
# Set before importing numerical libraries.
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = "1"

import hashlib
import h5py
import healpy as hp
import logging
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import astropy.units as u
from IPython.display import display
from scipy.integrate import quad
from astromodels import Cutoff_powerlaw, Parameter, clone_model, load_model
from astromodels.utils.configuration import astromodels_config
from threeML import DataList, JointLikelihood
from histpy import Histogram

from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.response import (
    FullDetectorResponse, BinnedInstrumentResponse,
    BinnedThreeMLPointSourceResponse, BinnedThreeMLModelFolding,
)
from cosipy.data_io import EmCDSBinnedData
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.statistics import PoissonLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.util import fetch_wasabi_file

logging.getLogger("cosipy").setLevel(logging.WARNING)


def display_tables(*tables):
    # Tables are rendered inline; no CSV/HTML export is performed.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        for table in tables:
            display(table)

14:14:12 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=432723;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=646219;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=90333;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=660314;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

14:14:13 WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=15094;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=598121;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

14:14:13 INFO      Starting 3ML!                                                                     ]8;id=712427;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=516334;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=522088;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=160409;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   ROOT minimizer not available                                                ]8;id=617982;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=639754;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1208\1208]8;;\

         WARNING   Multinest minimizer not available                                           ]8;id=626201;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=62092;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1218\1218]8;;\

         WARNING   PyGMO is not available                                                      ]8;id=816786;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=913429;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1228\1228]8;;\

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=94343;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=310487;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=51653;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=177018;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

In [2]:
# Works from the cosipy repository root or any directory beneath it.
COSI_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "cosipy").is_dir() and (p / "pyproject.toml").is_file()),
    Path.cwd(),
)
MODEL_PATH = (COSI_ROOT.parent / "AGN-HE-detectability" /
              "cosi_bat_157m_top50_models.yaml")
DATA_DIR = (Path.home() / "Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
            "Radio_Quiet_AGN/DC4_Files")
DOWNLOAD_DIR = COSI_ROOT / "agn_sensitivity_data"  # Missing reference files only.
CHECKPOINT_PATH = (COSI_ROOT / "agn_sensitivity_checkpoints" /
                   "bat157_top50_compact_results.csv")
SAVE_COMPACT_CHECKPOINT = True
TOTAL_BACKGROUND_PATH = (DATA_DIR / "Background" /
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck.hdf5")
FOV_MAX_OFFAXIS_DEG = 60.0
SOURCE_NAMES = None  # None = all 50; use a short list for a smoke test.
CUTOFF_VALUES_KEV = np.arange(100.0, 1001.0, 100.0)
N_REALIZATIONS = 20
RANDOM_SEED = 20260918
DOWNLOAD_DATA = False
BAT_BAND_KEV = (14.0, 195.0)
ENERGY_BAND_KEV = (100.0, 10000.0)
TARGET_MEDIAN_TS = 25.0
FOV_TARGET_DURATION_MONTHS = 3.0
NOMINAL_TRIAL_DURATION_MONTHS = 3.0
FOV_TARGET_EXPOSURE_SCALE = FOV_TARGET_DURATION_MONTHS / NOMINAL_TRIAL_DURATION_MONTHS
TS_THRESHOLD = 9.0  # Secondary 3-sigma reporting threshold.
EXPOSURE_SCALE_2YR = 8.0
BACKGROUND_FLOOR_COUNTS = 1e-12
KEV_TO_ERG = 1.602176634e-9

if not isinstance(SAVE_COMPACT_CHECKPOINT, bool):
    raise TypeError("SAVE_COMPACT_CHECKPOINT must be True or False")
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Generate the YAML first or update MODEL_PATH: {MODEL_PATH}")
if not isinstance(N_REALIZATIONS, int) or N_REALIZATIONS < 1:
    raise ValueError("N_REALIZATIONS must be a positive integer")
if not isinstance(RANDOM_SEED, int) or RANDOM_SEED < 0:
    raise ValueError("RANDOM_SEED must be a nonnegative integer")
if not np.isfinite(TARGET_MEDIAN_TS) or TARGET_MEDIAN_TS <= 0:
    raise ValueError("TARGET_MEDIAN_TS must be finite and positive")
if (not np.isfinite(FOV_TARGET_EXPOSURE_SCALE)
        or FOV_TARGET_EXPOSURE_SCALE <= 0):
    raise ValueError("The FOV target exposure scale must be finite and positive")
CUTOFF_VALUES_KEV = np.asarray(CUTOFF_VALUES_KEV, dtype=float)
if (CUTOFF_VALUES_KEV.ndim != 1 or len(CUTOFF_VALUES_KEV) == 0
        or not np.all(np.isfinite(CUTOFF_VALUES_KEV))
        or np.any(CUTOFF_VALUES_KEV <= 0)
        or np.any(np.diff(CUTOFF_VALUES_KEV) <= 0)):
    raise ValueError("CUTOFF_VALUES_KEV must be finite, positive, unique, and increasing")


## Load the YAML and construct the cutoff grid

The YAML stores the top-50 selection, sky positions, BAT photon indices, and
300 keV reference spectra. The BAT energy flux is integrated from each loaded
reference spectrum. For every trial cutoff, `K` is solved again to preserve
that flux. Positions, index, pivot, and cutoff stay fixed in the likelihood;
only `K` and the background rate are fitted.

In [3]:
def band_fluxes(shape, band):
    photon = quad(lambda energy: float(shape(energy)), *band,
                  epsabs=1e-30, epsrel=1e-9)[0]
    energy = quad(lambda value: value * float(shape(value)), *band,
                  epsabs=1e-30, epsrel=1e-9)[0]
    return photon, energy * KEV_TO_ERG


def amplitude_for_energy_flux(gamma, cutoff_keV, energy_flux, band=BAT_BAND_KEV):
    integral = quad(
        lambda energy: energy ** (1.0 - gamma) * np.exp(-energy / cutoff_keV),
        *band, epsabs=1e-30, epsrel=1e-11,
    )[0]
    if not np.isfinite(integral) or integral <= 0:
        raise ValueError("Invalid normalization integral")
    return energy_flux / (KEV_TO_ERG * integral)


def select_sources(catalog, names=None):
    available = list(catalog.point_sources)
    if catalog.extended_sources or not available:
        raise ValueError("Expected a YAML containing only point sources")
    selected = available if names is None else list(names)
    if not selected or len(selected) != len(set(selected)):
        raise ValueError("Choose at least one source, without duplicates")
    unknown = set(selected) - set(available)
    if unknown:
        raise ValueError(f"Unknown sources: {sorted(unknown)}; available: {available}")
    for name in selected:
        source = catalog.point_sources[name]
        if len(source.components) != 1 or not isinstance(
                source.spectrum.main.shape, Cutoff_powerlaw):
            raise ValueError(f"{name}: expected one Cutoff_powerlaw component")
        shape = source.spectrum.main.shape
        values = [shape.K.value, shape.index.value, shape.xc.value, shape.piv.value]
        if not np.all(np.isfinite(values)) or shape.K.value <= 0:
            raise ValueError(f"{name}: invalid reference spectrum")
    return selected


def isolated_model_at_cutoff(catalog, name, cutoff_keV):
    reference = catalog.point_sources[name].spectrum.main.shape
    gamma = -reference.index.value
    bat_flux = band_fluxes(reference, BAT_BAND_KEV)[1]
    amplitude = amplitude_for_energy_flux(gamma, cutoff_keV, bat_flux)

    model = clone_model(catalog)
    for other in list(model.point_sources):
        if other != name:
            model.remove_source(other)
    for parameter in model.parameters.values():
        parameter.free = False
    shape = model.point_sources[name].spectrum.main.shape
    shape.K.remove_transformation()
    shape.K.min_value = 0.0
    shape.K.max_value = None
    shape.xc.value = float(cutoff_keV)
    shape.K.value = amplitude
    shape.K.delta = 0.1 * amplitude
    shape.K.free = True
    return model, bat_flux


def scenario_table(catalog, names, cutoffs):
    rows = []
    catalog_order = list(catalog.point_sources)
    for name in names:
        reference = catalog.point_sources[name]
        shape = reference.spectrum.main.shape
        gamma = -shape.index.value
        bat_flux = band_fluxes(shape, BAT_BAND_KEV)[1]
        for cutoff in cutoffs:
            amplitude = amplitude_for_energy_flux(gamma, cutoff, bat_flux)
            photon_integral = quad(
                lambda energy: amplitude * energy ** (-gamma) * np.exp(-energy / cutoff),
                *ENERGY_BAND_KEV, epsabs=1e-30, epsrel=1e-9,
            )[0]
            energy_integral = quad(
                lambda energy: amplitude * energy ** (1.0 - gamma) * np.exp(-energy / cutoff),
                *ENERGY_BAND_KEV, epsabs=1e-30, epsrel=1e-9,
            )[0] * KEV_TO_ERG
            rows.append({
                "catalog_rank": catalog_order.index(name) + 1,
                "source": name,
                "ra_deg": reference.position.ra.value,
                "dec_deg": reference.position.dec.value,
                "gamma": gamma,
                "astromodel_index": shape.index.value,
                "cutoff_keV": float(cutoff),
                "K_ph_cm2_s_keV": amplitude,
                "bat_14_195_energy_flux_erg_cm2_s": bat_flux,
                "photon_flux_0p1_10_MeV_ph_cm2_s": photon_integral,
                "energy_flux_0p1_10_MeV_erg_cm2_s": energy_integral,
            })
    return pd.DataFrame(rows)


def skip_fainter_source_after_24mo_failure(source_index, failed_index):
    """Skip a fainter source after a brighter source fails at 24 months."""
    if failed_index is None:
        return False
    if not np.all(np.isfinite([source_index, failed_index])):
        raise ValueError("Source-pruning indices must be finite")
    return source_index <= failed_index


In [4]:
catalog_model = load_model(str(MODEL_PATH))
selected_names = select_sources(catalog_model, SOURCE_NAMES)
scenarios = scenario_table(catalog_model, selected_names, CUTOFF_VALUES_KEV)
maximum_trials = len(selected_names) * len(CUTOFF_VALUES_KEV) * N_REALIZATIONS
print(f"Maximum: {len(selected_names)} sources x {len(CUTOFF_VALUES_KEV)} cutoffs x "
      f"{N_REALIZATIONS} trials = {maximum_trials:,} likelihood trials")
print("Sources are processed in decreasing BAT flux.")
print("After a conclusive 24-month full-grid non-detection, later fainter sources are "
      "skipped unless their astromodels index is less negative.")
print(f"Each evaluated source stops after the first complete cutoff group whose estimated "
      f"{FOV_TARGET_DURATION_MONTHS:g}-month, {FOV_MAX_OFFAXIS_DEG:g}° FOV-cut "
      f"TS reaches {TARGET_MEDIAN_TS:g}.")
display_tables(scenarios)


Maximum: 50 sources x 10 cutoffs x 20 trials = 10,000 likelihood trials
Sources are processed in decreasing BAT flux.
After a conclusive 24-month full-grid non-detection, later fainter sources are skipped unless their astromodels index is less negative.
Each evaluated source stops after the first complete cutoff group whose estimated 3-month, 60° FOV-cut TS reaches 25.


,catalog_rank,source,ra_deg,dec_deg,gamma,astromodel_index,cutoff_keV,K_ph_cm2_s_keV,bat_14_195_energy_flux_erg_cm2_s,photon_flux_0p1_10_MeV_ph_cm2_s,energy_flux_0p1_10_MeV_erg_cm2_s
0,1,Cen_A,201.3651,-43.0191,1.87,-1.87,100.0,0.310402,1.181840e-09,0.000877,2.106148e-10
1,1,Cen_A,201.3651,-43.0191,1.87,-1.87,200.0,0.233402,1.181840e-09,0.001473,4.147973e-10
2,1,Cen_A,201.3651,-43.0191,1.87,-1.87,300.0,0.209944,1.181840e-09,0.001802,5.623198e-10
3,1,Cen_A,201.3651,-43.0191,1.87,-1.87,400.0,0.198656,1.181840e-09,0.002017,6.794417e-10
4,1,Cen_A,201.3651,-43.0191,1.87,-1.87,500.0,0.192028,1.181840e-09,0.002173,7.774134e-10
5,1,Cen_A,201.3651,-43.0191,1.87,-1.87,600.0,0.187670,1.181840e-09,0.002292,8.620756e-10
6,1,Cen_A,201.3651,-43.0191,1.87,-1.87,700.0,0.184588,1.181840e-09,0.002387,9.368824e-10
7,1,Cen_A,201.3651,-43.0191,1.87,-1.87,800.0,0.182293,1.181840e-09,0.002464,1.004062e-09
8,1,Cen_A,201.3651,-43.0191,1.87,-1.87,900.0,0.180518,1.181840e-09,0.002530,1.065144e-09
9,1,Cen_A,201.3651,-43.0191,1.87,-1.87,1000.0,0.179105,1.181840e-09,0.002585,1.121225e-09


## Obtain the DC4 continuum files

These are the same continuum response, smoothed background, checksums, and
three-month spacecraft history used by the existing AGN notebook. Existing
files are reused. With `DOWNLOAD_DATA = True`, missing reference files are
downloaded to `DOWNLOAD_DIR`; calculation outputs are never written.

In [5]:
FILES = {
    "orientation": (
        "Orientation/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits",
        "ca94ff1d7a73c1f41479aaf598807673",
    ),
    "response": (
        "Responses/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5",
        "7121f094be50e7bfe9b31e53015b0e85",
    ),
    "background": (
        "Backgrounds/gal_DC4_bkg_continuum_binned_smoothed_fwhm_10.0deg.hdf5",
        "1acceb619b0d7e43eea8cb29ffe74279",
    ),
}
paths = {}
for key, (remote, _) in FILES.items():
    filename = Path(remote).name
    candidates = (DATA_DIR / filename, DATA_DIR / "Background" / filename,
                  DOWNLOAD_DIR / filename)
    paths[key] = next((path for path in candidates if path.is_file()), DOWNLOAD_DIR / filename)
    print(f"{key}: {paths[key]}")
for key, (remote, checksum) in FILES.items():
    if DOWNLOAD_DATA and not paths[key].is_file():
        paths[key].parent.mkdir(parents=True, exist_ok=True)
        fetch_wasabi_file("COSI-SMEX/DC4/Data/" + remote,
                          output=paths[key], checksum=checksum)
    elif not paths[key].is_file():
        raise FileNotFoundError(paths[key])
    else:
        # Streaming avoids loading a large response into memory just to hash it.
        with paths[key].open("rb") as handle:
            actual = hashlib.file_digest(handle, "md5").hexdigest()
        if actual != checksum:
            raise ValueError(f"Checksum mismatch: {paths[key]}")

orientation: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits
response: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5
background: /Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files/gal_DC4_bkg_continuum_binned_smoothed_fwhm_10.0deg.hdf5


## Check the response and background

Use every native response/background bin spanning 100–10,000 keV. Exactly
empty background bins receive the configured tiny count floor in both
simulation and fitting. The diagnostics report signal predicted in those bins.

In [6]:
def regularize_background(histogram, floor):
    background = histogram.project("Em", "Phi", "PsiChi").to_dense()
    values = np.asarray(background.contents, dtype=float).copy()
    if not np.all(np.isfinite(values)) or np.any(values < 0) or values.sum() <= 0:
        raise ValueError("Background must contain finite nonnegative counts with positive total")
    if not np.isfinite(floor) or floor <= 0:
        raise ValueError("The background floor must be finite and positive")
    empty = values == 0
    values[empty] = floor
    background[:] = values
    return background, empty


def check_response_axes(detector_response, background, band):
    for label, axis in (("Ei", detector_response.axes["Ei"]),
                        ("Em", detector_response.measurement_axes["Em"]),
                        ("background Em", background.axes["Em"])):
        edges = axis.edges.to_value(u.keV)
        if not np.all(np.diff(edges) > 0) or not np.allclose(edges[[0, -1]], band):
            raise ValueError(f"{label} must span {band} keV; got {edges[[0, -1]]}")
    for label in ("Em", "Phi"):
        if background.axes[label] != detector_response.measurement_axes[label]:
            raise ValueError(f"Background/response {label} axes do not match")
    if background.axes["PsiChi"].coordsys is None:
        raise ValueError("Background PsiChi axis needs a coordinate system")


def make_folding(model, detector_response, background, orientation):
    # Only axes enter the response construction. Trial counts go into a fresh
    # PoissonLikelihood; no private response data attributes need to be changed.
    template_data = EmCDSBinnedData(background)
    instrument = BinnedInstrumentResponse(detector_response, template_data)
    psr = BinnedThreeMLPointSourceResponse(
        data=template_data, instrument_response=instrument, sc_history=orientation,
        energy_axis=detector_response.axes["Ei"],
        polarization_axis=(detector_response.axes["Pol"]
                           if "Pol" in detector_response.axes.labels else None),
        nside=2 * background.axes["PsiChi"].nside,
    )
    folding = BinnedThreeMLModelFolding(data=template_data, point_source_response=psr)
    folding.set_model(model)
    signal = folding.expectation()
    if signal.axes != background.axes:
        raise ValueError("Folded signal and background axes do not agree")
    values = np.asarray(signal.contents, dtype=float)
    if not np.all(np.isfinite(values)) or np.any(values < 0) or values.sum() <= 0:
        raise ValueError("Folded signal must be finite, nonnegative and nonzero")
    return folding, signal

In [7]:
sc_orientation = SpacecraftHistory.open(paths["orientation"])
background, empty_background_bins = regularize_background(
    Histogram.open(paths["background"]), BACKGROUND_FLOOR_COUNTS,
)
livetime_s = float(sc_orientation.cumulative_livetime().to_value(u.s))
if not np.isfinite(livetime_s) or livetime_s <= 0:
    raise ValueError("Spacecraft history must have positive finite livetime")
background_rate_hz = float(np.asarray(background.contents).sum()) / livetime_s
with FullDetectorResponse.open(paths["response"]) as detector_response:
    check_response_axes(detector_response, background, ENERGY_BAND_KEV)
print(f"Livetime: {livetime_s:.3f} s; background rate: {background_rate_hz:.6g} Hz")
print(f"Empty background bins receiving a floor: {empty_background_bins.sum()}")
print("Likelihood and flux-reporting range: 0.1–10 MeV")

Livetime: 6579555.000 s; background rate: 25.6289 Hz
Empty background bins receiving a floor: 11530
Likelihood and flux-reporting range: 0.1–10 MeV


## Prepare the 60° FOV-cut estimate

The calculation follows the existing AGN tutorial. It folds each renormalized
source/cutoff spectrum through the response with the 15-second pointings and
Earth occultation, before and after an off-axis cut measured from spacecraft
+Z. The total time-resolved background allocates its roughly one-hour bins by
selected livetime fraction; those energy-dependent fractions scale the smooth
background spectrum used by the likelihood.

For a background-dominated count estimate at fixed observing duration,

$$G\approx f_S/\sqrt{f_B},\qquad
F_{\min,\mathrm{cut}}/F_{\min,\mathrm{uncut}}\approx1/G.$$

The large input files are reduced here. During the sequential likelihood
search, the deterministic FOV gains are calculated for the current source and
used to test the three-month stopping condition. Only source/cutoff pairs that
were actually evaluated appear in the result tables. All projections and
result tables remain in memory. This approximation omits angular likelihood
information, fitted-background effects, and unresolved background changes
within a coarse time bin. It does not apply the cut to the Poisson trials and is
not a cut likelihood refit.


In [8]:
def interval_integrals(edges, amounts, query):
    """Integrate interval-constant rates, including partial interval overlaps."""
    edges, amounts = np.asarray(edges), np.asarray(amounts)
    if len(edges) != len(amounts) + 1 or np.any(np.diff(edges) <= 0):
        raise ValueError("Expected increasing interval edges and one amount per interval")
    return np.interp(query, edges, np.r_[0.0, np.cumsum(amounts)],
                     left=0.0, right=np.sum(amounts))


def reduce_fov_inputs(total_path, smooth_path, response_path, band):
    """Read DC4 files into small in-memory projections; never write a disk cache."""
    for path in (total_path, smooth_path, response_path):
        if not Path(path).is_file():
            raise FileNotFoundError(path)
    with h5py.File(total_path, "r") as file:
        shape = tuple(file["hist/contents/shape"][...])
        labels = [file[f"hist/axes/{i}"].attrs["label"] for i in range(4)]
        if labels != ["Time", "Em", "Phi", "PsiChi"]:
            raise ValueError(f"Unexpected background axis order: {labels}")
        if file["hist/contents/fill_value"][()] != 0:
            raise ValueError("Expected zero-filled sparse DC4 background")
        time_edges = file["hist/axes/0"][...]
        em_edges = file["hist/axes/1"][...]
        counts_te = np.zeros(shape[0] * shape[1])
        coords, values = file["hist/contents/coords"], file["hist/contents/data"]
        # Read a bounded chunk of the large sparse histogram at a time.
        for start in range(0, len(values), 2_000_000):
            indices = coords[:2, start:start + 2_000_000]
            counts_te += np.bincount(
                indices[0] * shape[1] + indices[1],
                weights=values[start:start + 2_000_000], minlength=counts_te.size,
            )
        counts_te = counts_te.reshape(shape[:2])
    with h5py.File(smooth_path, "r") as file:
        np.testing.assert_array_equal(em_edges, file["hist/axes/1"][...])
        if file["hist/contents/fill_value"][()] != 0:
            raise ValueError("Expected zero-filled sparse smoothed background")
        smooth_em = np.bincount(file["hist/contents/coords"][1],
                                weights=file["hist/contents/data"][...],
                                minlength=len(em_edges) - 1)
    with h5py.File(response_path, "r") as file:
        labels = [file[f"DRM/AXES/{i}"].attrs["label"] for i in range(5)]
        if labels != ["NuLambda", "Ei", "Em", "Phi", "PsiChi"]:
            raise ValueError(f"Expected the unpolarized DC4 continuum response: {labels}")
        np.testing.assert_array_equal(em_edges, file["DRM/AXES/2"][...])
        ei_edges = file["DRM/AXES/1"][...]
        nside = int(file["DRM/AXES/0"].attrs["nside"])
        nest = str(file["DRM/AXES/0"].attrs["scheme"]).lower() == "nested"
        counts = file["DRM/COUNTS"]
        area = file["DRM/EFF_AREA"][...] * u.Unit(file["DRM"].attrs["UNIT"]).to(u.cm**2)
        response = np.zeros(counts.shape[:3])
        for pixel in range(counts.shape[0]):
            response[pixel] = counts[pixel].sum(axis=(2, 3), dtype=float) * area[:, None]
    for edges in (ei_edges, em_edges):
        if not np.all(np.diff(edges) > 0) or not np.allclose(edges[[0, -1]], band):
            raise ValueError(f"FOV response/background must span {band} keV")
    if np.any(np.diff(time_edges) <= 0):
        raise ValueError("Background time edges must increase")
    for values in (counts_te, smooth_em, response):
        if not np.all(np.isfinite(values)) or np.any(values < 0):
            raise ValueError("Nonfinite or negative background/response values")
    if np.any(counts_te.sum(axis=0) <= 0) or np.any(smooth_em <= 0):
        raise ValueError("Each measured-energy bin needs positive background counts")
    print("FOV input projections ready in memory; no cache files written.")
    return dict(counts_te=counts_te, time_edges=time_edges, em_edges=em_edges,
                smooth_em=smooth_em, ei_edges=ei_edges, response=response,
                nside=nside, nest=nest)



def estimate_fov_gain_scenarios(catalog, evaluated, orientation, inputs,
                                max_offaxis_deg):
    """Count-based FOV estimate for source/cutoff pairs actually evaluated."""
    required = {"source", "cutoff_keV"}
    if not required.issubset(evaluated.columns):
        raise ValueError(f"Evaluated scenarios need columns {sorted(required)}")
    evaluated = evaluated[["source", "cutoff_keV"]].drop_duplicates()
    if evaluated.empty:
        raise ValueError("At least one evaluated source/cutoff pair is required")
    if not np.isfinite(max_offaxis_deg) or not 0 < max_offaxis_deg <= 180:
        raise ValueError("FOV maximum off-axis angle must be in (0, 180] degrees")
    counts_te, time_edges, smooth_em = (
        inputs[key] for key in ("counts_te", "time_edges", "smooth_em")
    )
    response, ei_edges, em_edges = (
        inputs[key] for key in ("response", "ei_edges", "em_edges")
    )
    times = orientation.obstime.unix
    livetime = orientation.livetime.to_value(u.s)
    duration = np.diff(times)
    if np.any(duration <= 0) or np.any(livetime < 0) or np.any(livetime > duration + 1e-5):
        raise ValueError("Invalid orientation durations or livetimes")
    complete = ((time_edges[:-1] >= times[0] - 1e-5)
                & (time_edges[1:] <= times[-1] + 1e-5))
    covered = np.flatnonzero(complete)
    if not len(covered):
        raise ValueError("No complete background time bin is covered by the orientation")
    low, high = time_edges[covered[0]], time_edges[covered[-1] + 1]
    common_fraction = np.maximum(
        0, np.minimum(times[1:], high) - np.maximum(times[:-1], low),
    ) / duration
    common_live = livetime * common_fraction
    background_live = np.diff(interval_integrals(times, livetime, time_edges))
    if np.any(complete & (background_live <= 0) & (counts_te.sum(axis=1) > 0)):
        raise ValueError("Background counts in covered bins with zero instrument livetime")
    common_counts = counts_te[complete].sum(axis=0)
    if np.any(common_counts <= 0):
        raise ValueError("Common interval lacks counts in a background energy bin")
    smooth_common = smooth_em * common_counts / counts_te.sum(axis=0)

    rows, energy_rows = [], []
    for name, source_scenarios in evaluated.groupby("source", sort=False):
        if name not in catalog.point_sources:
            raise ValueError(f"Unknown evaluated source: {name}")
        source = catalog.point_sources[name]
        coord = source.position.sky_coord
        local = orientation.get_target_in_sc_frame(coord)
        theta = np.pi / 2 - local.lat.to_value(u.rad)[:-1]
        phi = local.lon.to_value(u.rad)[:-1]
        selected = theta <= np.deg2rad(max_offaxis_deg)
        visible = orientation.get_source_visibility(coord).to_value(u.s) * common_fraction
        pixels, weights = hp.get_interp_weights(
            inputs["nside"], theta, phi, nest=inputs["nest"],
        )
        exposure = np.bincount(
            pixels.ravel(), weights=(weights * visible[None]).ravel(),
            minlength=response.shape[0],
        )
        cut_exposure = np.bincount(
            pixels.ravel(), weights=(weights * (visible * selected)[None]).ravel(),
            minlength=response.shape[0],
        )
        selected_live = np.diff(interval_integrals(
            times, livetime * selected, time_edges,
        ))
        fraction = np.divide(
            selected_live, background_live, out=np.zeros_like(background_live),
            where=background_live > 0,
        )
        if np.any((fraction < -1e-8) | (fraction > 1 + 1e-8)):
            raise ValueError("Computed background retention outside [0, 1]")
        fraction = np.clip(fraction, 0, 1)
        background_fraction_em = (
            counts_te[complete] * fraction[complete, None]
        ).sum(axis=0) / common_counts
        background_cut = smooth_common * background_fraction_em

        for cutoff in source_scenarios["cutoff_keV"].to_numpy(dtype=float):
            model, _ = isolated_model_at_cutoff(catalog, name, cutoff)
            spectrum = model.point_sources[name].spectrum.main.shape
            incident_flux = np.array([
                quad(lambda energy: float(spectrum(energy)), low_edge, high_edge,
                     epsabs=1e-35, epsrel=1e-9)[0]
                for low_edge, high_edge in zip(ei_edges[:-1], ei_edges[1:])
            ])
            signal = np.einsum("p,pij,i->j", exposure, response, incident_flux)
            signal_cut = np.einsum(
                "p,pij,i->j", cut_exposure, response, incident_flux,
            )
            del model
            f_signal = signal_cut.sum() / signal.sum() if signal.sum() > 0 else np.nan
            f_background = background_cut.sum() / smooth_common.sum()
            gain = f_signal / np.sqrt(f_background) if f_background > 0 else np.nan
            valid = np.isfinite(gain) and gain > 0
            rows.append({
                "source": name, "cutoff_keV": float(cutoff),
                "max_offaxis_deg": max_offaxis_deg,
                "signal_counts_no_cut": signal.sum(),
                "signal_counts_cut": signal_cut.sum(),
                "background_counts_no_cut": smooth_common.sum(),
                "background_counts_cut": background_cut.sum(),
                "livetime_fraction": np.sum(common_live * selected) / common_live.sum(),
                "signal_fraction": f_signal,
                "background_fraction": f_background,
                "gain": gain,
                "flux_threshold_ratio": 1 / gain if valid else np.nan,
                "flux_threshold_reduction_percent": (
                    100 * (1 - 1 / gain) if valid else np.nan
                ),
                "status": "ok" if valid else "no usable selected exposure",
            })
            for index in range(len(em_edges) - 1):
                energy_rows.append({
                    "source": name, "cutoff_keV": float(cutoff),
                    "energy_min_keV": em_edges[index],
                    "energy_max_keV": em_edges[index + 1],
                    "signal": signal[index], "signal_cut": signal_cut[index],
                    "background": smooth_common[index],
                    "background_cut": background_cut[index],
                })
    coverage = pd.DataFrame([{
        "common_start_unix": low, "common_end_unix": high,
        "common_livetime_s": common_live.sum(),
        "background_time_bin_s": np.median(np.diff(time_edges)),
        "excluded_background_counts": counts_te.sum() - common_counts.sum(),
        "excluded_background_percent": 100 * (1 - common_counts.sum() / counts_te.sum()),
    }])
    return pd.DataFrame(rows), pd.DataFrame(energy_rows), coverage


In [9]:
fov_inputs = reduce_fov_inputs(
    TOTAL_BACKGROUND_PATH, paths["background"], paths["response"], ENERGY_BAND_KEV,
)
print("FOV source folding is deferred until the early-stopping search is complete.")


FOV input projections ready in memory; no cache files written.
FOV source folding is deferred until the early-stopping search is complete.


## Sequential source/cutoff Poisson trials

For each source, cutoffs are evaluated in increasing order. For each evaluated
source/cutoff pair, the deterministic folded expectation is cached for all
realizations. Source-plus-background and background-only hypotheses are fit to
the same Poisson draw. The source normalization has no logarithmic transform,
so the null fit can set exactly `K = 0`. Failed fits and materially negative or
nonfinite TS values remain explicit.

After all realizations at one cutoff finish, the notebook computes their median
three-month uncut TS and converts it to an estimated three-month, 60° FOV-cut TS:

$$TS_{\rm 3mo,FOV\ est}=G^2\,\mathrm{median}(TS_{\rm 3mo,uncut}).$$

Early stopping occurs only when every likelihood fit is valid, the FOV estimate
is usable, and this three-month estimate is at least `TARGET_MEDIAN_TS`.
Otherwise the next cutoff is tested.


Sources are ordered by decreasing BAT flux. A pruning boundary is established
only when a source conclusively remains below TS 25 across the full cutoff grid
after scaling its FOV-cut TS to 24 months. Subsequent fainter sources with an
equal or more negative index are skipped; sources with a less negative index are
still evaluated. A source detectable in the 24-month extrapolation, fit failures,
or invalid FOV estimates do not establish a pruning boundary. The source-run
decision table records the outcome and reference source for every skip.


In [10]:
def draw_trial(signal, background, rng):
    if signal.axes != background.axes:
        raise ValueError("Signal/background axes do not match")
    counts = rng.poisson(np.asarray(signal.contents, dtype=float))
    counts += rng.poisson(np.asarray(background.contents, dtype=float))
    histogram = Histogram(background.axes)
    histogram[:] = counts
    return EmCDSBinnedData(histogram)


def checked_ts(loglike_signal, loglike_null, tolerance=1e-4):
    if not np.all(np.isfinite([loglike_signal, loglike_null])):
        raise ValueError("Nonfinite fitted likelihood")
    ts_raw = 2 * (loglike_signal - loglike_null)
    if ts_raw < -tolerance:
        raise ValueError(f"Materially negative TS: {ts_raw:.6g}")
    return max(0.0, ts_raw), ts_raw


def fit_without_transforms(analysis):
    # threeML clones the fitted model when building its results. astromodels
    # restores Cutoff_powerlaw's default log transform in that clone, which
    # otherwise rejects our zero lower bound even after remove_transformation().
    # Disable transformation checks during fitting/result construction only;
    # restore the user's setting even if a fit raises. Run trials sequentially.
    previous = astromodels_config.modeling.use_parameter_transforms
    try:
        astromodels_config.modeling.use_parameter_transforms = False
        # A results-only clone may internally represent exact zero as log10(0).
        # Suppress only that known warning; retain all other numerical warnings.
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="divide by zero encountered in _log10",
                                    category=RuntimeWarning)
            analysis.fit(compute_covariance=False, quiet=True, n_samples=1)
    finally:
        astromodels_config.modeling.use_parameter_transforms = previous


def fit_trial(model, folding, signal, background, orientation, rate_hz, rng):
    source = next(iter(model.point_sources.values()))
    shape = source.spectrum.main.shape
    catalog_K = shape.K.value
    result = {"status": "failed", "error": "", "ts": np.nan, "ts_raw": np.nan,
              "loglike_signal": np.nan, "loglike_null": np.nan,
              "K_fit_ph_cm2_s_keV": np.nan, "background_rate_fit_hz": np.nan,
              "background_rate_null_hz": np.nan}
    plugin = None
    try:
        shape.K.value, shape.K.free = catalog_K, True
        data = draw_trial(signal, background, rng)
        bkg = FreeNormBinnedBackground({"Background": background},
                                      sc_history=orientation, copy=True)
        likelihood = PoissonLikelihood(data, folding, bkg)
        plugin = ThreeMLPluginInterface("cosi", likelihood, folding, bkg)
        # FreeNormBinnedBackground normalizes the template to unit integral.
        # Its amplitude is counts/livetime in Hz, not a dimensionless scale.
        plugin.bkg_parameter["Background"] = Parameter(
            "Background", rate_hz, min_value=0, max_value=None,
            delta=0.05 * rate_hz, unit=u.Hz,
        )
        analysis = JointLikelihood(model, DataList(plugin), verbose=False)
        analysis.set_minimizer("minuit")
        fit_without_transforms(analysis)
        result["loglike_signal"] = float(plugin.get_log_like())
        result["K_fit_ph_cm2_s_keV"] = shape.K.value
        result["background_rate_fit_hz"] = plugin.bkg_parameter["Background"].value

        shape.K.value, shape.K.free = 0.0, False
        fit_without_transforms(analysis)
        result["loglike_null"] = float(plugin.get_log_like())
        result["background_rate_null_hz"] = plugin.bkg_parameter["Background"].value
        result["ts"], result["ts_raw"] = checked_ts(
            result["loglike_signal"], result["loglike_null"],
        )
        result["status"] = "ok"
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        shape.K.value, shape.K.free = catalog_K, True
        # JointLikelihood attaches its nuisance parameters to the model.
        # Remove them so the next trial starts with a clean source-only model.
        if plugin is not None:
            for parameter in plugin.nuisance_parameters.values():
                if parameter.name in model:
                    model.remove_external_parameter(parameter.name)
    return result


def summarize_trials(trials, flux_table, exposure_scale=8.0, threshold=9.0):
    summaries = []
    for source in flux_table.to_dict("records"):
        group = trials.loc[
            (trials["source"] == source["source"])
            & (trials["cutoff_keV"] == source["cutoff_keV"])
        ]
        valid = group.loc[(group["status"] == "ok") & np.isfinite(group["ts"]), "ts"]
        median = float(valid.median()) if len(valid) else np.nan
        failures = len(group) - len(valid)
        complete = len(group) > 0 and failures == 0
        summaries.append({
            **source, "n_trials": len(group), "n_valid": len(valid), "n_failed": failures,
            "ts_threshold": threshold,
            "median_ts_3mo": median, "approx_local_sigma_3mo": np.sqrt(median),
            "median_ts_2yr_approx": exposure_scale * median,
            "approx_local_sigma_2yr": np.sqrt(exposure_scale * median),
            "passes_ts_threshold_3mo": bool(median >= threshold) if complete else None,
            "passes_ts_threshold_2yr_approx": bool(exposure_scale * median >= threshold) if complete else None,
            "classification_status": "complete" if complete else "inconclusive_fit_failures",
        })
    return pd.DataFrame(summaries)

In [11]:
# Results remain in memory. Rerunning replaces these variables.
run_metadata = {
    "model_path": str(MODEL_PATH.resolve()),
    "model_sha256": hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    "source_names": selected_names,
    "cutoff_values_keV": CUTOFF_VALUES_KEV.tolist(),
    "target_median_ts": TARGET_MEDIAN_TS,
    "stopping_statistic": "estimated three-month TS after 60-degree FOV cut",
    "fov_target_duration_months": FOV_TARGET_DURATION_MONTHS,
    "nominal_trial_duration_months": NOMINAL_TRIAL_DURATION_MONTHS,
    "fov_target_exposure_scale": FOV_TARGET_EXPOSURE_SCALE,
    "early_stopping_requires_all_trials_valid": True,
    "source_pruning_rule": ("after a conclusive 24-month full-grid "
                            "non-detection, skip fainter sources unless index "
                            "is less negative"),
    "n_realizations_per_scenario": N_REALIZATIONS,
    "random_seed": RANDOM_SEED,
    "energy_band_keV": ENERGY_BAND_KEV,
    "bat_normalization_band_keV": BAT_BAND_KEV,
    "livetime_s": livetime_s,
    "exposure_scale_2yr": EXPOSURE_SCALE_2YR,
    "secondary_ts_threshold": TS_THRESHOLD,
    "save_compact_checkpoint": SAVE_COMPACT_CHECKPOINT,
    "checkpoint_path": str(CHECKPOINT_PATH.resolve()),
    "background_floor_counts": BACKGROUND_FLOOR_COUNTS,
    "background_rate_hz": background_rate_hz,
    "files": {key: {"path": str(paths[key]), "md5": checksum}
              for key, (_, checksum) in FILES.items()},
}
all_trials, diagnostics = [], []
fov_estimate_parts, fov_energy_parts = [], []
source_run_decisions = []
hardest_failed_index = None
hardest_failed_source = None
fov_coverage = None
catalog_order = list(catalog_model.point_sources)
with FullDetectorResponse.open(paths["response"]) as detector_response:
    check_response_axes(detector_response, background, ENERGY_BAND_KEV)
    for name in selected_names:
        source_index = catalog_order.index(name)
        spectral_index = float(
            catalog_model.point_sources[name].spectrum.main.shape.index.value
        )
        if skip_fainter_source_after_24mo_failure(spectral_index, hardest_failed_index):
            source_run_decisions.append({
                "catalog_rank": source_index + 1, "source": name,
                "astromodel_index": spectral_index, "run_status": "skipped",
                "n_cutoffs_tested": 0, "target_reached": False,
                "target_reached_24mo_est": None,
                "max_ts_24mo_fov_est": np.nan,
                "pruning_reference_source": hardest_failed_source,
                "pruning_reference_index": hardest_failed_index,
                "reason": ("fainter than a conclusive 24-month non-detection "
                           "and index is equal or more negative"),
            })
            print(
                f"Skipping {name}: index={spectral_index:g} is not less negative "
                f"than 24-month-undetected brighter source {hardest_failed_source} "
                f"(index={hardest_failed_index:g}).", flush=True,
            )
            continue
        source_reached_target = False
        source_all_scenarios_conclusive = True
        source_reaches_24mo_target = False
        source_max_ts_24mo_fov = -np.inf
        source_cutoffs_tested = 0

        # Geometry and exposure are common to all cutoffs for this source. Fold
        # the deterministic spectra once here, then retain rows only for cutoff
        # values actually reached by the sequential likelihood search.
        source_scenario_grid = scenarios.loc[
            scenarios["source"] == name, ["source", "cutoff_keV"]
        ]
        source_fov, source_fov_energy, source_coverage = estimate_fov_gain_scenarios(
            catalog_model, source_scenario_grid, sc_orientation,
            fov_inputs, FOV_MAX_OFFAXIS_DEG,
        )
        if fov_coverage is None:
            fov_coverage = source_coverage
        else:
            pd.testing.assert_frame_equal(fov_coverage, source_coverage)

        for cutoff_index, cutoff in enumerate(CUTOFF_VALUES_KEV):
            source_cutoffs_tested += 1
            print(f"Preparing {name}, Ecut={cutoff:g} keV "
                  f"({N_REALIZATIONS} trials)...", flush=True)
            model, bat_flux = isolated_model_at_cutoff(catalog_model, name, cutoff)
            folding, signal = make_folding(
                model, detector_response, background, sc_orientation,
            )
            source_counts = np.asarray(signal.contents, dtype=float)
            in_empty = float(source_counts[empty_background_bins].sum())
            diagnostics.append({
                "source": name,
                "cutoff_keV": float(cutoff),
                "expected_source_counts_3mo": float(source_counts.sum()),
                "n_floored_background_bins": int(empty_background_bins.sum()),
                "expected_signal_counts_in_floored_bins": in_empty,
            })
            if in_empty > 0.1:
                print(f"  Review background: {in_empty:.3g} expected signal "
                      "counts in floored bins.")
            rng = np.random.default_rng(np.random.SeedSequence(
                [RANDOM_SEED, source_index, cutoff_index],
            ))
            cutoff_results = []
            for trial in range(N_REALIZATIONS):
                result = fit_trial(
                    model, folding, signal, background, sc_orientation,
                    background_rate_hz, rng,
                )
                row = {
                    "source": name,
                    "cutoff_keV": float(cutoff),
                    "trial": trial,
                    **result,
                }
                all_trials.append(row)
                cutoff_results.append(row)
                print(f"  {trial + 1}/{N_REALIZATIONS}: {result['status']}; "
                      f"TS={result['ts']:.3g}", flush=True)
                if result["status"] != "ok":
                    print("   ", result["error"])
            del folding, signal, model

            this_fov = source_fov.loc[
                source_fov["cutoff_keV"] == float(cutoff)
            ].copy()
            this_energy = source_fov_energy.loc[
                source_fov_energy["cutoff_keV"] == float(cutoff)
            ].copy()
            if len(this_fov) != 1:
                raise ValueError(f"Expected one FOV estimate for {name} at {cutoff:g} keV")
            fov_estimate_parts.append(this_fov)
            fov_energy_parts.append(this_energy)
            fov_row = this_fov.iloc[0]

            valid_ts = [row["ts"] for row in cutoff_results
                        if row["status"] == "ok" and np.isfinite(row["ts"])]
            cutoff_complete = len(valid_ts) == N_REALIZATIONS
            cutoff_median_ts = float(np.median(valid_ts)) if valid_ts else np.nan
            fov_valid = (fov_row["status"] == "ok"
                         and np.isfinite(fov_row["gain"])
                         and fov_row["gain"] > 0)
            median_ts_3mo_fov = (
                FOV_TARGET_EXPOSURE_SCALE * cutoff_median_ts * fov_row["gain"] ** 2
                if fov_valid and np.isfinite(cutoff_median_ts) else np.nan
            )
            source_all_scenarios_conclusive &= cutoff_complete and fov_valid
            if cutoff_complete and fov_valid and np.isfinite(median_ts_3mo_fov):
                ts_24mo_fov = EXPOSURE_SCALE_2YR * median_ts_3mo_fov
                source_max_ts_24mo_fov = max(
                    source_max_ts_24mo_fov, ts_24mo_fov,
                )
                source_reaches_24mo_target |= ts_24mo_fov >= TARGET_MEDIAN_TS
            print(f"  Median TS (3mo, uncut)={cutoff_median_ts:.3g}; "
                  f"G={fov_row['gain']:.3g}; "
                  f"TS ({FOV_TARGET_DURATION_MONTHS:g}mo, FOV est)="
                  f"{median_ts_3mo_fov:.3g}; "
                  f"valid trials={len(valid_ts)}/{N_REALIZATIONS}", flush=True)
            if (cutoff_complete and fov_valid
                    and median_ts_3mo_fov >= TARGET_MEDIAN_TS):
                source_reached_target = True
                print(f"  {name} reached estimated {FOV_TARGET_DURATION_MONTHS:g}-month "
                      f"FOV-cut TS >= {TARGET_MEDIAN_TS:g} at Ecut={cutoff:g} keV; "
                      "moving to the next source.", flush=True)
                break
        max_ts_24mo = (
            source_max_ts_24mo_fov
            if np.isfinite(source_max_ts_24mo_fov) else np.nan
        )
        if source_reached_target:
            source_run_decisions.append({
                "catalog_rank": source_index + 1, "source": name,
                "astromodel_index": spectral_index, "run_status": "detected_3mo",
                "n_cutoffs_tested": source_cutoffs_tested, "target_reached": True,
                "target_reached_24mo_est": True,
                "max_ts_24mo_fov_est": max_ts_24mo,
                "pruning_reference_source": None, "pruning_reference_index": np.nan,
                "reason": "reached the three-month FOV-cut TS target",
            })
        elif source_all_scenarios_conclusive and source_reaches_24mo_target:
            print(f"  {name} did not reach the three-month target, but reaches "
                  f"the approximate 24-month FOV-cut TS target; no pruning "
                  "boundary was added.", flush=True)
            source_run_decisions.append({
                "catalog_rank": source_index + 1, "source": name,
                "astromodel_index": spectral_index,
                "run_status": "detected_24mo_only",
                "n_cutoffs_tested": source_cutoffs_tested, "target_reached": False,
                "target_reached_24mo_est": True,
                "max_ts_24mo_fov_est": max_ts_24mo,
                "pruning_reference_source": None, "pruning_reference_index": np.nan,
                "reason": "below target at 3 months but reaches it at 24 months",
            })
        elif source_all_scenarios_conclusive:
            print(f"  {name} conclusively did not reach FOV-cut TS >= "
                  f"{TARGET_MEDIAN_TS:g} at any cutoff even in the approximate "
                  "24-month exposure.", flush=True)
            source_run_decisions.append({
                "catalog_rank": source_index + 1, "source": name,
                "astromodel_index": spectral_index,
                "run_status": "not_detected_24mo_full_grid",
                "n_cutoffs_tested": source_cutoffs_tested, "target_reached": False,
                "target_reached_24mo_est": False,
                "max_ts_24mo_fov_est": max_ts_24mo,
                "pruning_reference_source": None, "pruning_reference_index": np.nan,
                "reason": "conclusive non-detection through 24 months across full grid",
            })
            if (hardest_failed_index is None
                    or spectral_index > hardest_failed_index):
                hardest_failed_index = spectral_index
                hardest_failed_source = name
        else:
            print(f"  {name} remained inconclusive because at least one cutoff "
                  "had failed fits or an invalid FOV estimate; no pruning "
                  "boundary was added.", flush=True)
            source_run_decisions.append({
                "catalog_rank": source_index + 1, "source": name,
                "astromodel_index": spectral_index, "run_status": "inconclusive",
                "n_cutoffs_tested": source_cutoffs_tested, "target_reached": False,
                "target_reached_24mo_est": None,
                "max_ts_24mo_fov_est": max_ts_24mo,
                "pruning_reference_source": None, "pruning_reference_index": np.nan,
                "reason": "fit failure or invalid FOV estimate",
            })

diagnostics = pd.DataFrame(diagnostics)
fov_estimates = pd.concat(fov_estimate_parts, ignore_index=True)
fov_energy_counts = pd.concat(fov_energy_parts, ignore_index=True)
tested_scenarios = scenarios.merge(
    diagnostics[["source", "cutoff_keV"]],
    on=["source", "cutoff_keV"], how="inner", validate="one_to_one",
)
trials = pd.DataFrame(all_trials)
summary = summarize_trials(
    trials, tested_scenarios, EXPOSURE_SCALE_2YR, TS_THRESHOLD,
).merge(
    diagnostics,
    on=["source", "cutoff_keV"], validate="one_to_one",
).merge(
    fov_estimates[["source", "cutoff_keV", "gain", "status"]].rename(
        columns={"status": "fov_status"}
    ),
    on=["source", "cutoff_keV"], validate="one_to_one",
)
stop_valid = ((summary["classification_status"] == "complete")
              & (summary["fov_status"] == "ok")
              & np.isfinite(summary["gain"]) & (summary["gain"] > 0))
summary["median_ts_3mo_fov_est"] = (
    FOV_TARGET_EXPOSURE_SCALE * summary["median_ts_3mo"] * summary["gain"] ** 2
).where(stop_valid)
summary["sigma_3mo_fov_est"] = np.sqrt(summary["median_ts_3mo_fov_est"])
summary["reaches_target_3mo_fov_est"] = pd.Series(
    np.where(stop_valid,
             summary["median_ts_3mo_fov_est"] >= TARGET_MEDIAN_TS,
             None),
    dtype="boolean",
)
failed_trials = trials.loc[trials["status"] != "ok"]

# This is the only calculated output written by the notebook. It contains one
# row per evaluated source/cutoff pair and is sufficient for the final detection
# tables and plots. Trial draws, fitted parameters per realization, generated
# source models, FOV energy counts, plots, and response products are not saved.
checkpoint_columns = [
    "catalog_rank", "source", "astromodel_index", "cutoff_keV",
    "n_trials", "n_valid", "n_failed",
    "classification_status", "median_ts_3mo", "gain", "fov_status",
    "median_ts_3mo_fov_est", "sigma_3mo_fov_est",
]
compact_checkpoint = summary[checkpoint_columns].copy()
compact_checkpoint["target_median_ts"] = TARGET_MEDIAN_TS
compact_checkpoint["fov_max_offaxis_deg"] = FOV_MAX_OFFAXIS_DEG
compact_checkpoint["nominal_trial_duration_months"] = NOMINAL_TRIAL_DURATION_MONTHS
compact_checkpoint["random_seed"] = RANDOM_SEED
compact_checkpoint["n_realizations_configured"] = N_REALIZATIONS
compact_checkpoint["model_sha256"] = run_metadata["model_sha256"]
if SAVE_COMPACT_CHECKPOINT:
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_checkpoint = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + ".tmp")
    compact_checkpoint.to_csv(temporary_checkpoint, index=False)
    temporary_checkpoint.replace(CHECKPOINT_PATH)
    print(f"Compact plot checkpoint saved: {CHECKPOINT_PATH}")

source_run_decisions = pd.DataFrame(source_run_decisions)
decision_by_source = source_run_decisions.set_index("source")
stop_rows = []
for name in selected_names:
    decision = decision_by_source.loc[name]
    group = summary.loc[summary["source"] == name].sort_values("cutoff_keV")
    if group.empty:
        stop_rows.append({
            "catalog_rank": int(decision["catalog_rank"]),
            "source": name,
            "astromodel_index": float(decision["astromodel_index"]),
            "run_status": decision["run_status"],
            "n_cutoffs_tested": 0,
            "last_cutoff_tested_keV": np.nan,
            "target_reached": False,
            "target_reached_24mo_est": decision["target_reached_24mo_est"],
            "max_ts_24mo_fov_est": decision["max_ts_24mo_fov_est"],
            "cutoff_at_target_keV": np.nan,
            "median_ts_3mo_uncut_at_stop_or_grid_end": np.nan,
            "gain_at_stop_or_grid_end": np.nan,
            "ts_3mo_fov_est_at_stop_or_grid_end": np.nan,
            "sigma_3mo_fov_est_at_stop_or_grid_end": np.nan,
            "stop_reason": decision["reason"],
        })
        continue
    reached = group.loc[group["reaches_target_3mo_fov_est"].fillna(False)]
    final = reached.iloc[0] if len(reached) else group.iloc[-1]
    stop_rows.append({
        "catalog_rank": int(final["catalog_rank"]),
        "source": name,
        "astromodel_index": float(final["astromodel_index"]),
        "run_status": decision["run_status"],
        "n_cutoffs_tested": len(group),
        "last_cutoff_tested_keV": float(group["cutoff_keV"].max()),
        "target_reached": bool(len(reached)),
        "target_reached_24mo_est": decision["target_reached_24mo_est"],
        "max_ts_24mo_fov_est": decision["max_ts_24mo_fov_est"],
        "cutoff_at_target_keV": float(final["cutoff_keV"]) if len(reached) else np.nan,
        "median_ts_3mo_uncut_at_stop_or_grid_end": float(final["median_ts_3mo"]),
        "gain_at_stop_or_grid_end": float(final["gain"]),
        "ts_3mo_fov_est_at_stop_or_grid_end": float(final["median_ts_3mo_fov_est"]),
        "sigma_3mo_fov_est_at_stop_or_grid_end": float(final["sigma_3mo_fov_est"]),
        "stop_reason": decision["reason"],
    })
stopping_summary = pd.DataFrame(stop_rows)

print("Evaluated source/cutoff summary")
display_tables(summary.sort_values(["catalog_rank", "cutoff_keV"]))
print("Source-run decisions, including flux/index pruning")
display_tables(source_run_decisions.sort_values("catalog_rank"))
print("Per-source stopping summary")
display_tables(stopping_summary.sort_values("catalog_rank"))
print(f"Executed {len(trials):,} of at most {maximum_trials:,} likelihood trials.")
print(f"Failed trials: {len(failed_trials)} of {len(trials)}")
if len(failed_trials):
    display_tables(failed_trials)
print("Run configuration")
display_tables(pd.DataFrame([
    {"setting": key, "value": str(value)} for key, value in run_metadata.items()
]))


Preparing Cen_A, Ecut=100 keV (20 trials)...


14:14:42 INFO      set the minimizer to minuit                                             ]8;id=573245;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=342430;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to MINUIT                                             ]8;id=213432;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=706580;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1034\1034]8;;\

  1/20: ok; TS=6.23


         INFO      set the minimizer to minuit                                             ]8;id=706949;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=452360;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

...

  19/20: ok; TS=0.364


14:25:50 INFO      set the minimizer to minuit                                             ]8;id=655860;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=150704;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

         INFO      set the minimizer to MINUIT                                             ]8;id=604500;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=201652;file:///Users/parshadkp/.pyenv/versions/3.12.8/envs/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1034\1034]8;;\

  20/20: ok; TS=0.922
  Median TS (3mo, uncut)=4.01; G=1.87; TS (3mo, FOV est)=14; valid trials=20/20
Preparing NGC_4388, Ecut=300 keV (20 trials)...


KeyboardInterrupt: 

## Three-month FOV stopping result and two-year significance

The sequential search uses

$$TS_{\rm 3mo,FOV\ est}\approx
G^2\,\mathrm{median}(TS_{\rm 3mo,uncut})$$

and stops at the first cutoff where this estimate reaches 25. The FOV estimate
uses the same three-month exposure as the likelihood trials. It assumes the
count-based FOV gain adequately describes the change in significance.

For the separately reported two-year extrapolation,

$$TS_{2\,\mathrm{yr}}\approx8\,TS_{3\,\mathrm{mo}},\qquad Z\approx\sqrt{TS}.$$

The two-year factor scales nominal calendar duration. It is not a two-year
orbit simulation and can fail for variable sources, changing backgrounds,
sparse counts, or systematics. Twenty realizations give limited median-TS
precision. Rows with failed fits or unusable FOV estimates are inconclusive and
cannot trigger early stopping. The cutoff changes the extrapolated 0.1–10 MeV
flux while every evaluated scenario retains the same measured BAT-band flux.


In [ ]:
def significance_matrix(frame, column):
    matrix = frame.pivot(index="source", columns="cutoff_keV", values=column)
    order = frame.groupby("source")[column].max().sort_values(ascending=False).index
    return matrix.reindex(index=order, columns=CUTOFF_VALUES_KEV)


cutoff_summary = summary.groupby("cutoff_keV", as_index=False).agg(
    n_sources_evaluated=("source", "size"),
    n_complete=("classification_status", lambda values: (values == "complete").sum()),
    n_reached_target_3mo_fov=("reaches_target_3mo_fov_est",
                              lambda values: values.fillna(False).sum()),
    median_ts_3mo_uncut=("median_ts_3mo", "median"),
    median_ts_3mo_fov_est=("median_ts_3mo_fov_est", "median"),
    median_sigma_3mo_fov_est=("sigma_3mo_fov_est", "median"),
)
print("Evaluated population by imposed cutoff")
display_tables(cutoff_summary)

sigma3 = significance_matrix(summary, "approx_local_sigma_3mo")
sigma6fov = significance_matrix(summary, "sigma_3mo_fov_est")
positive = np.concatenate([
    sigma3.to_numpy()[np.isfinite(sigma3.to_numpy()) & (sigma3.to_numpy() > 0)],
    sigma6fov.to_numpy()[np.isfinite(sigma6fov.to_numpy()) & (sigma6fov.to_numpy() > 0)],
])
norm = LogNorm(vmin=max(positive.min(), 1e-2), vmax=positive.max()) if len(positive) else None
fig, axes = plt.subplots(1, 2, figsize=(16, 14), sharey=True)
for axis, matrix, title in (
    (axes[0], sigma3, "Three-month uncut trials"),
    (axes[1], sigma6fov, "Three-month 60° FOV estimate"),
):
    image = axis.imshow(matrix.to_numpy(), aspect="auto", interpolation="nearest",
                        cmap="viridis", norm=norm)
    axis.set_xticks(np.arange(len(matrix.columns)), [f"{value:g}" for value in matrix.columns],
                    rotation=45, ha="right")
    axis.set_yticks(np.arange(len(matrix.index)), matrix.index)
    axis.set(title=title, xlabel="Imposed cutoff (keV)")
axes[0].set_ylabel("BAT Seyfert candidate (ordered by maximum significance)")
fig.colorbar(image, ax=axes, label="Approximate significance (sigma)", shrink=0.7)
fig.suptitle("Early-stopped BAT Seyfert-proxy cutoff search · 0.1–10 MeV")
fig.subplots_adjust(top=0.94, bottom=0.08, left=0.18, right=0.92, wspace=0.08)
plt.show()

ts3fov = significance_matrix(summary, "median_ts_3mo_fov_est")
fig, ax = plt.subplots(figsize=(10, 7))
for source, row in ts3fov.iterrows():
    tested = row.dropna()
    ax.plot(tested.index, tested.values, marker="o", alpha=0.25, linewidth=1)
ax.axhline(TARGET_MEDIAN_TS, color="tab:red", linestyle="--",
           label=f"Stopping target: estimated three-month FOV TS = {TARGET_MEDIAN_TS:g}")
ax.set(xlabel="Imposed cutoff (keV)",
       ylabel="Estimated three-month TS after 60° FOV cut",
       title="Evaluated cutoff trajectory for each source")
ax.set_yscale("symlog", linthresh=1)
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cutoff_summary.cutoff_keV, cutoff_summary.n_sources_evaluated, "o-",
        label="Sources evaluated")
ax.plot(cutoff_summary.cutoff_keV,
        cutoff_summary.n_reached_target_3mo_fov, "s-",
        label=f"Reached estimated three-month FOV TS >= {TARGET_MEDIAN_TS:g}")
ax.set(xlabel="Imposed cutoff (keV)", ylabel="Number of sources",
       title="Early-stopping progress across the cutoff grid")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()


## FOV-cut significance details

The source/cutoff-specific FOV gains used by the stopping loop are displayed
below. For any duration scaled from the three-month likelihood trials,

$$TS_{\mathrm{FOV\ est}}\approx
(\mathrm{exposure\ scale})\,G^2\,\mathrm{median}(TS_{\mathrm{3mo,uncut}}).$$

The three-month stopping value uses an exposure scale of one. The two-year value
uses a scale of eight and remains an additional projection. Incomplete trial
groups or unusable FOV exposure are marked inconclusive and receive no post-cut
significance. These values are count-based estimates, not likelihood fits to
FOV-selected data.


In [ ]:
def estimate_cut_significance(uncut_summary, estimates,
                              exposure_scale_2yr=8.0,
                              detection_threshold=9.0, target_threshold=25.0):
    combined = uncut_summary.merge(
        estimates[["source", "cutoff_keV", "max_offaxis_deg", "gain", "status"]],
        on=["source", "cutoff_keV"], how="left", validate="one_to_one",
        suffixes=("", "_estimate"),
    )
    status_column = "status_estimate" if "status_estimate" in combined else "status"
    gain_column = "gain_estimate" if "gain_estimate" in combined else "gain"
    if combined[status_column].isna().any():
        raise ValueError("Missing an FOV estimate for an evaluated source/cutoff scenario")
    valid = ((combined["n_trials"] > 0)
             & (combined["n_valid"] == combined["n_trials"])
             & (combined["n_failed"] == 0)
             & (combined[status_column] == "ok")
             & np.isfinite(combined["median_ts_3mo"])
             & (combined["median_ts_3mo"] >= 0)
             & np.isfinite(combined[gain_column]) & (combined[gain_column] > 0))
    result = combined[[
        "catalog_rank", "source", "cutoff_keV", "max_offaxis_deg",
        "n_trials", "n_valid", "n_failed",
    ]].copy()
    result["gain"] = combined[gain_column]
    result["sigma_3mo_uncut"] = np.sqrt(combined["median_ts_3mo"])
    result["ts_3mo_fov_est"] = (
        combined["median_ts_3mo"] * result["gain"] ** 2
    ).where(valid)
    result["ts_2yr_fov_est"] = exposure_scale_2yr * result["ts_3mo_fov_est"]
    for period in ("3mo", "2yr"):
        result[f"sigma_{period}_fov_est"] = np.sqrt(result[f"ts_{period}_fov_est"])
        result[f"passes_ts9_{period}_fov_est"] = pd.Series(
            np.where(valid, result[f"ts_{period}_fov_est"] >= detection_threshold, None),
            dtype="boolean",
        )
        result[f"reaches_target_{period}_fov_est"] = pd.Series(
            np.where(valid, result[f"ts_{period}_fov_est"] >= target_threshold, None),
            dtype="boolean",
        )
    result["sigma_2yr_uncut_approx"] = (
        np.sqrt(exposure_scale_2yr) * result["sigma_3mo_uncut"]
    )
    result["interpretation"] = np.where(
        valid, "count-based estimate; no cut refit",
        "inconclusive: inspect trials/FOV status",
    )
    return result


print("FOV estimates for evaluated source/cutoff pairs")
display_tables(fov_estimates.sort_values(["source", "cutoff_keV"]))
print("Common background/orientation coverage")
display_tables(fov_coverage)

fov_significance = estimate_cut_significance(
    summary, fov_estimates, EXPOSURE_SCALE_2YR, TS_THRESHOLD, TARGET_MEDIAN_TS,
)
np.testing.assert_allclose(
    fov_significance["ts_3mo_fov_est"], summary["median_ts_3mo_fov_est"],
    rtol=1e-12, atol=0, equal_nan=True,
)
print("Estimated post-cut significance for evaluated source/cutoff pairs")
display_tables(fov_significance.sort_values(["catalog_rank", "cutoff_keV"]))

fov_population = fov_significance.groupby("cutoff_keV", as_index=False).agg(
    n_evaluated=("source", "size"),
    n_target_3mo_fov=("reaches_target_3mo_fov_est",
                      lambda values: values.fillna(False).sum()),
    n_target_2yr_fov=("reaches_target_2yr_fov_est",
                      lambda values: values.fillna(False).sum()),
)
display_tables(fov_population)

fov_matrix = fov_significance.pivot(
    index="source", columns="cutoff_keV", values="sigma_3mo_fov_est",
).reindex(index=significance_matrix(summary, "sigma_3mo_fov_est").index,
          columns=CUTOFF_VALUES_KEV)
values = fov_matrix.to_numpy()
positive = values[np.isfinite(values) & (values > 0)]
norm = None
if len(positive):
    lower, upper = max(positive.min(), 1e-2), positive.max()
    norm = LogNorm(vmin=lower, vmax=max(upper, lower * 1.000001))
fig, ax = plt.subplots(figsize=(10, 14))
image = ax.imshow(values, aspect="auto", interpolation="nearest",
                  cmap="viridis", norm=norm)
ax.set_xticks(np.arange(len(fov_matrix.columns)),
              [f"{value:g}" for value in fov_matrix.columns], rotation=45, ha="right")
ax.set_yticks(np.arange(len(fov_matrix.index)), fov_matrix.index)
ax.set(xlabel="Imposed cutoff (keV)", ylabel="BAT source",
       title=f"Estimated three-month significance after {FOV_MAX_OFFAXIS_DEG:g}° FOV cut")
fig.colorbar(image, ax=ax, label="Approximate significance (sigma)")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for period, marker, label in (
    ("3mo", "o", "Three months (stopping statistic)"),
    ("2yr", "^", "Two-year approximation"),
):
    ax.plot(fov_population.cutoff_keV,
            fov_population[f"n_target_{period}_fov"], marker + "-", label=label)
ax.set(xlabel="Imposed cutoff (keV)",
       ylabel=f"Evaluated sources with FOV-estimated TS >= {TARGET_MEDIAN_TS:g}",
       title="Target significance after the FOV estimate")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()


## Detected sources at the TS ≥ 25 threshold

This final cell is self-contained. After the search has run once and written the
compact checkpoint, it can be run by itself in a fresh kernel. It loads only the
compact evaluated-scenario CSV and does not repeat response folding, Poisson
simulations, or likelihood fits.

It displays separate tables for the three-month FOV estimate and the approximate
24-month FOV estimate. For each period, a source appears once at its first
evaluated cutoff satisfying the threshold. The 24-month estimate assumes a
steady source and comparable exposure and background:

$$TS_{24\,\mathrm{mo,FOV\ est}}\approx
8\,TS_{3\,\mathrm{mo,FOV\ est}}.$$

The checkpoint does not contain trial-level fits, generated source files,
response projections, FOV energy-count tables, or saved plots.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the default checkpoint even when this is the only cell run after a
# kernel restart. An explicitly defined CHECKPOINT_PATH still takes precedence.
if "COSI_ROOT" not in globals():
    COSI_ROOT = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / "cosipy").is_dir() and (p / "pyproject.toml").is_file()),
        Path.cwd(),
    )
if "CHECKPOINT_PATH" not in globals():
    CHECKPOINT_PATH = (COSI_ROOT / "agn_sensitivity_checkpoints" /
                       "bat157_top50_compact_results.csv")

if "summary" in globals():
    plot_summary = summary.copy()
    plot_target_ts = float(TARGET_MEDIAN_TS)
    plot_fov_deg = float(FOV_MAX_OFFAXIS_DEG)
    plot_trial_months = float(NOMINAL_TRIAL_DURATION_MONTHS)
    results_origin = "current kernel"
else:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(
            f"Compact checkpoint not found: {CHECKPOINT_PATH}. Run the sequential "
            "source/cutoff search once to create it."
        )
    plot_summary = pd.read_csv(CHECKPOINT_PATH)
    metadata_columns = {
        "target_median_ts", "fov_max_offaxis_deg",
        "nominal_trial_duration_months",
    }
    missing_metadata = metadata_columns - set(plot_summary.columns)
    if missing_metadata:
        raise ValueError(
            f"Checkpoint is missing metadata columns: {sorted(missing_metadata)}"
        )
    for column in metadata_columns:
        values = plot_summary[column].dropna().unique()
        if len(values) != 1:
            raise ValueError(f"Checkpoint must contain one unique {column}")
    plot_target_ts = float(plot_summary["target_median_ts"].iloc[0])
    plot_fov_deg = float(plot_summary["fov_max_offaxis_deg"].iloc[0])
    plot_trial_months = float(plot_summary["nominal_trial_duration_months"].iloc[0])
    results_origin = str(CHECKPOINT_PATH)

required_columns = {
    "catalog_rank", "source", "cutoff_keV", "n_trials", "n_valid", "n_failed",
    "classification_status", "median_ts_3mo", "gain", "fov_status",
    "median_ts_3mo_fov_est",
}
missing_columns = required_columns - set(plot_summary.columns)
if missing_columns:
    raise ValueError(f"Results are missing columns: {sorted(missing_columns)}")


def first_threshold_crossings(frame, months, exposure_scale, threshold):
    """Return each source's first evaluated cutoff reaching the scaled FOV TS."""
    valid = ((frame["classification_status"] == "complete")
             & (frame["fov_status"] == "ok")
             & np.isfinite(frame["median_ts_3mo_fov_est"])
             & (frame["median_ts_3mo_fov_est"] >= 0))
    candidates = frame.loc[valid, [
        "catalog_rank", "source", "cutoff_keV", "n_trials", "n_valid", "n_failed",
        "median_ts_3mo", "gain", "median_ts_3mo_fov_est",
    ]].copy()
    candidates["duration_months"] = float(months)
    candidates["ts_fov_est"] = exposure_scale * candidates["median_ts_3mo_fov_est"]
    candidates["sigma_fov_est"] = np.sqrt(candidates["ts_fov_est"])
    candidates = candidates.loc[candidates["ts_fov_est"] >= threshold]
    candidates = candidates.sort_values(["catalog_rank", "cutoff_keV"])
    first = candidates.groupby("source", as_index=False, sort=False).first()
    return first[[
        "catalog_rank", "source", "duration_months", "cutoff_keV",
        "median_ts_3mo", "gain", "ts_fov_est", "sigma_fov_est",
        "n_trials", "n_valid", "n_failed",
    ]].sort_values(["cutoff_keV", "catalog_rank"]).reset_index(drop=True)


MONTHS_3 = 3.0
MONTHS_24 = 24.0
SCALE_3 = MONTHS_3 / plot_trial_months
SCALE_24 = MONTHS_24 / plot_trial_months
if not np.isclose(SCALE_3, 1.0) or not np.isclose(SCALE_24, 8.0):
    raise ValueError("Expected three- and 24-month exposure scales of 1 and 8")

detected_sources_3mo = first_threshold_crossings(
    plot_summary, MONTHS_3, SCALE_3, plot_target_ts,
)
detected_sources_24mo = first_threshold_crossings(
    plot_summary, MONTHS_24, SCALE_24, plot_target_ts,
)

print(f"Results loaded from: {results_origin}")
print(
    f"Three-month detections after the {plot_fov_deg:g}° FOV cut "
    f"(TS >= {plot_target_ts:g}): {len(detected_sources_3mo)}"
)
display(detected_sources_3mo)
print(
    f"Approximate 24-month detections after the {plot_fov_deg:g}° FOV cut "
    f"(TS >= {plot_target_ts:g}): {len(detected_sources_24mo)}"
)
display(detected_sources_24mo)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for axis, detected, title in (
    (axes[0], detected_sources_3mo, "Three months"),
    (axes[1], detected_sources_24mo, "24-month approximation"),
):
    if detected.empty:
        axis.text(0.5, 0.5, "No sources reached the threshold",
                  ha="center", va="center", transform=axis.transAxes)
        axis.set_axis_off()
        continue
    plot_data = detected.sort_values("ts_fov_est")
    bars = axis.barh(
        plot_data["source"], plot_data["ts_fov_est"],
        color="tab:blue", alpha=0.85,
    )
    axis.axvline(
        plot_target_ts, color="tab:red", linestyle="--", linewidth=1.5,
        label=f"TS = {plot_target_ts:g}",
    )
    for bar, cutoff in zip(bars, plot_data["cutoff_keV"]):
        axis.annotate(
            f"{cutoff:g} keV",
            xy=(bar.get_width(), bar.get_y() + bar.get_height() / 2),
            xytext=(4, 0), textcoords="offset points",
            va="center", fontsize=8,
        )
    axis.set(
        xlabel=f"Estimated TS after {plot_fov_deg:g}° FOV cut",
        ylabel="Source",
        title=f"{title}: {len(detected)} sources",
    )
    axis.set_xlim(left=0)
    axis.grid(axis="x", alpha=0.2)
    axis.legend()
fig.suptitle(f"Sources reaching FOV-estimated TS ≥ {plot_target_ts:g}")
fig.tight_layout()
plt.show()
